# 01 — ممیزی داده و کنترل کیفیت دیتاست CHARGED

## هدف نوت‌بوک

این نوت‌بوک ساختار و کیفیت خام دیتاست CHARGED را بدون تغییر در داده‌ها بررسی می‌کند. هدف‌ها شامل شناسایی فایل‌های موجود، شهرهای پوشش‌داده‌شده، متغیرها، انواع داده، مقادیر گمشده، رکوردهای تکراری و سازگاری زمانی است.

داده‌های شهرهای آمستردام، ژوهانسبورگ، لس‌آنجلس، ملبورن و سائوپائولو برای آموزش مدل استفاده خواهند شد. تمام داده‌های شنژن در این مرحله شناسایی و از مجموعهٔ آموزش کنار گذاشته می‌شوند تا برای اعتبارسنجی خارجی با UrbanEV هیچ هم‌پوشانی شهری ایجاد نشود.

هیچ رکوردی در این نوت‌بوک حذف یا اصلاح نمی‌شود. تمام تصمیم‌های پاک‌سازی پس از مستندسازی نتایج ممیزی اتخاذ خواهند شد.

In [3]:
# Prompt: Import required libraries and define reproducible project paths.

from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]

PROJECT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "charged" / "Hourly.zip").exists()
    ),
    None,
)

assert PROJECT_ROOT is not None, (
    "Hourly.zip پیدا نشد. بررسی کن فایل در مسیر "
    "data/raw/charged/Hourly.zip قرار داشته باشد."
)

RAW_CHARGED_DIR = PROJECT_ROOT / "data" / "raw" / "charged"
CHARGED_ARCHIVE_PATH = RAW_CHARGED_DIR / "Hourly.zip"

OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"CHARGED archive: {CHARGED_ARCHIVE_PATH.name}")
print(f"Archive size: {CHARGED_ARCHIVE_PATH.stat().st_size / (1024 ** 2):.2f} MB")

Project root: F:\UrbanEV_Charging_Demand
CHARGED archive: Hourly.zip
Archive size: 256.27 MB


In [4]:
# Prompt: Inspect the internal file structure of the CHARGED hourly archive without extracting it.

with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
    archive_files = charged_archive.namelist()

archive_structure = pd.DataFrame(
    {
        "file_path": archive_files,
        "file_extension": [Path(file_name).suffix for file_name in archive_files],
    }
)

print(f"Number of files inside Hourly.zip: {len(archive_structure):,}")

display(archive_structure.head(30))

Number of files inside Hourly.zip: 132


,file_path,file_extension
0,JHB/,
1,JHB/chargers.csv,.csv
2,JHB/distance.csv,.csv
3,JHB/duration.csv,.csv
4,JHB/e_price.csv,.csv
5,JHB/info.csv,.csv
6,JHB/poi.csv,.csv
7,JHB/sites.csv,.csv
8,JHB/s_price.csv,.csv
9,JHB/volume.csv,.csv


## ۱-۱. فهرست فایل‌های خام و شناسایی شهرهای مطالعه

در این مرحله، پوشه‌های خام هر شهر و فایل‌های موجود در آن‌ها بررسی می‌شوند. نسخه‌های `remove_zero` استفاده نخواهند شد، زیرا پیش‌پردازش آن‌ها پیش از ممیزی مستقل انجام شده است.

داده‌های شنژن (`SZH`) فقط برای بررسی ساختار فایل‌ها نگهداری می‌شوند و در آموزش یا تنظیم مدل وارد نخواهند شد.

In [5]:
# Prompt: Create an inventory of original CHARGED city folders and their available files.

EXPECTED_CITY_FILES = [
    "chargers.csv",
    "distance.csv",
    "duration.csv",
    "e_price.csv",
    "info.csv",
    "poi.csv",
    "sites.csv",
    "s_price.csv",
    "volume.csv",
    "weather.csv",
]

base_city_codes = sorted(
    {
        file_path.split("/")[0]
        for file_path in archive_files
        if "/" in file_path
        and "_remove_zero" not in file_path
        and file_path.split("/")[0]
    }
)

training_city_codes = [city for city in base_city_codes if city != "SZH"]
external_city_code = "SZH"

print("All cities:", base_city_codes)
print("Training cities:", training_city_codes)
print("Excluded city:", external_city_code)

archive_file_set = set(archive_files)
inventory_rows = []

with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
    for city in base_city_codes:
        for file_name in EXPECTED_CITY_FILES:
            archive_path = f"{city}/{file_name}"
            is_available = archive_path in archive_file_set

            file_size_mb = np.nan
            if is_available:
                file_size_mb = charged_archive.getinfo(archive_path).file_size / (1024 ** 2)

            inventory_rows.append(
                {
                    "city_code": city,
                    "dataset_role": "external_only" if city == "SZH" else "training_candidate",
                    "file_name": file_name,
                    "available": is_available,
                    "uncompressed_size_mb": file_size_mb,
                }
            )

charged_inventory = pd.DataFrame(inventory_rows)

inventory_summary = (
    charged_inventory
    .groupby(["city_code", "dataset_role"], as_index=False)
    .agg(
        available_files=("available", "sum"),
        total_uncompressed_size_mb=("uncompressed_size_mb", "sum"),
    )
    .sort_values("city_code")
)

display(inventory_summary)

charged_inventory.to_csv(
    OUTPUT_TABLES_DIR / "charged_raw_archive_inventory.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: outputs/tables/charged_raw_archive_inventory.csv")

All cities: ['AMS', 'JHB', 'LOA', 'MEL', 'SPO', 'SZH']
Training cities: ['AMS', 'JHB', 'LOA', 'MEL', 'SPO']
Excluded city: SZH


,city_code,dataset_role,available_files,total_uncompressed_size_mb
0,AMS,training_candidate,10,390.8604
1,JHB,training_candidate,10,9.8549
2,LOA,training_candidate,10,35.9076
3,MEL,training_candidate,10,38.5639
4,SPO,training_candidate,10,10.5094
5,SZH,external_only,10,290.6006


Saved: outputs/tables/charged_raw_archive_inventory.csv


## ۱-۲. بررسی ساختار فایل‌های مکانی ایستگاه‌ها

فایل‌های `info.csv` و `sites.csv` برای هر شهر بررسی می‌شوند تا متغیرهای قابل‌انتقال، مانند شناسهٔ ایستگاه، مختصات، ظرفیت و ویژگی‌های مکانی شناسایی شوند. این بررسی فقط ساختار فایل‌ها را می‌خواند و هیچ تغییری در داده‌های خام ایجاد نمی‌کند.

In [6]:
# Prompt: Compare the schema and sample records of station metadata files across all CHARGED cities.

from io import TextIOWrapper

METADATA_FILES = ["info.csv", "sites.csv"]

metadata_schema_rows = []
metadata_samples = {}

with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
    for city in base_city_codes:
        for file_name in METADATA_FILES:
            archive_path = f"{city}/{file_name}"

            with charged_archive.open(archive_path) as raw_file:
                sample_df = pd.read_csv(raw_file, nrows=5)

            metadata_samples[(city, file_name)] = sample_df.copy()

            metadata_schema_rows.append(
                {
                    "city_code": city,
                    "dataset_role": "external_only" if city == "SZH" else "training_candidate",
                    "file_name": file_name,
                    "sample_rows_read": len(sample_df),
                    "column_count": len(sample_df.columns),
                    "columns": " | ".join(sample_df.columns.astype(str)),
                }
            )

metadata_schema = pd.DataFrame(metadata_schema_rows)

display(metadata_schema)

metadata_schema.to_csv(
    OUTPUT_TABLES_DIR / "charged_station_metadata_schema.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Sample: AMS/info.csv")
display(metadata_samples[("AMS", "info.csv")])

print("Sample: AMS/sites.csv")
display(metadata_samples[("AMS", "sites.csv")])

print("Saved: outputs/tables/charged_station_metadata_schema.csv")

,city_code,dataset_role,file_name,sample_rows_read,column_count,columns
0,AMS,training_candidate,info.csv,1,9,city | country | abbreviation | total_chargers...
1,AMS,training_candidate,sites.csv,5,9,site_id | longitude | latitude | charger_num |...
2,JHB,training_candidate,info.csv,1,9,city | country | abbreviation | total_chargers...
3,JHB,training_candidate,sites.csv,5,9,site_id | longitude | latitude | charger_num |...
4,LOA,training_candidate,info.csv,1,8,city | country | abbreviation | total_chargers...
5,LOA,training_candidate,sites.csv,5,9,site_id | longitude | latitude | charger_num |...
6,MEL,training_candidate,info.csv,1,8,city | country | abbreviation | total_chargers...
7,MEL,training_candidate,sites.csv,5,9,site | longitude | latitude | charger_num | to...
8,SPO,training_candidate,info.csv,1,8,city | country | abbreviation | total_chargers...
9,SPO,training_candidate,sites.csv,5,9,site | longitude | latitude | charger_num | to...


Sample: AMS/info.csv


,city,country,abbreviation,total_chargers,total_sites,DBSCAN_eps,total_duration,total_volume,avg_power
0,Amsterdam,Netherlands,AMS,3526,2449,47.3629,"28,808,415.0873","239,175,391.2916",3.5160


Sample: AMS/sites.csv


,site_id,longitude,latitude,charger_num,total_duration,total_volume,avg_power,perimeter,area
0,0,4.8353,52.4059,1,0.0000,0.0000,0.0000,399.4622,"11,599.8034"
1,1,4.8586,52.4099,1,0.0000,0.0000,0.0000,399.4868,"11,600.8597"
2,2,4.8771,52.3399,1,0.0000,0.0000,0.0000,399.0581,"11,582.4889"
3,3,4.9086,52.3696,2,0.0000,0.0000,0.0000,399.2399,"11,590.2804"
4,4,4.9452,52.3051,1,0.0000,0.0000,0.0000,398.8457,"11,573.3806"


Saved: outputs/tables/charged_station_metadata_schema.csv


## ۱-۳. ممیزی ایستگاه‌ها و کیفیت متغیرهای مکانی

در این مرحله، فایل‌های `sites.csv` همهٔ شهرها بررسی می‌شوند. تمرکز بر تعداد ایستگاه‌ها و شارژرها، اعتبار مختصات، داده‌های گمشده، مختصات تکراری و میزان تقاضای صفر است.

متغیر `total_duration` شاخص اصلی تقاضای شارژ در این پژوهش خواهد بود. `total_volume` و `avg_power` نیز برای تحلیل مکمل کیفیت داده نگهداری می‌شوند.

In [7]:
# Prompt: Audit site-level spatial metadata and charging-demand fields across all CHARGED cities.

def read_csv_from_charged_archive(city_code, file_name):
    archive_path = f"{city_code}/{file_name}"

    with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
        with charged_archive.open(archive_path) as raw_file:
            return pd.read_csv(raw_file)


site_level_tables = []

for city in base_city_codes:
    city_sites = read_csv_from_charged_archive(city, "sites.csv").copy()

    site_identifier_column = (
        "site_id" if "site_id" in city_sites.columns else "site"
    )

    city_sites = city_sites.rename(
        columns={site_identifier_column: "site_id"}
    )

    city_sites["city_code"] = city
    city_sites["dataset_role"] = (
        "external_only" if city == external_city_code else "training_candidate"
    )

    site_level_tables.append(city_sites)

charged_sites = pd.concat(site_level_tables, ignore_index=True)

site_level_audit = (
    charged_sites
    .groupby(["city_code", "dataset_role"], as_index=False)
    .agg(
        site_count=("site_id", "nunique"),
        total_chargers=("charger_num", "sum"),
        missing_longitude=("longitude", lambda series: series.isna().sum()),
        missing_latitude=("latitude", lambda series: series.isna().sum()),
        invalid_longitude=(
            "longitude",
            lambda series: ((series < -180) | (series > 180)).sum(),
        ),
        invalid_latitude=(
            "latitude",
            lambda series: ((series < -90) | (series > 90)).sum(),
        ),
        duplicate_coordinates=(
            "site_id",
            lambda series: 0,
        ),
        zero_total_duration=(
            "total_duration",
            lambda series: (series == 0).sum(),
        ),
        zero_total_volume=(
            "total_volume",
            lambda series: (series == 0).sum(),
        ),
        missing_total_duration=(
            "total_duration",
            lambda series: series.isna().sum(),
        ),
        missing_total_volume=(
            "total_volume",
            lambda series: series.isna().sum(),
        ),
    )
)

duplicate_coordinate_counts = (
    charged_sites
    .duplicated(subset=["city_code", "longitude", "latitude"], keep=False)
    .groupby(charged_sites["city_code"])
    .sum()
    .rename("duplicate_coordinates")
    .reset_index()
)

site_level_audit = (
    site_level_audit
    .drop(columns="duplicate_coordinates")
    .merge(duplicate_coordinate_counts, on="city_code", how="left")
    .sort_values("city_code")
)

site_level_audit["zero_duration_percent"] = (
    100
    * site_level_audit["zero_total_duration"]
    / site_level_audit["site_count"]
)

site_level_audit["zero_volume_percent"] = (
    100
    * site_level_audit["zero_total_volume"]
    / site_level_audit["site_count"]
)

display(site_level_audit)

site_level_audit.to_csv(
    OUTPUT_TABLES_DIR / "charged_site_level_quality_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Combined site-level shape: {charged_sites.shape}")
print("Saved: outputs/tables/charged_site_level_quality_audit.csv")

,city_code,dataset_role,site_count,total_chargers,missing_longitude,missing_latitude,invalid_longitude,invalid_latitude,zero_total_duration,zero_total_volume,missing_total_duration,missing_total_volume,duplicate_coordinates,zero_duration_percent,zero_volume_percent
0,AMS,training_candidate,2449,3526,0,0,0,0,1061,1061,0,0,0,43.3238,43.3238
1,JHB,training_candidate,47,61,0,0,0,0,12,12,0,0,0,25.5319,25.5319
2,LOA,training_candidate,229,506,0,0,0,0,5,5,0,0,0,2.1834,2.1834
3,MEL,training_candidate,63,64,0,0,0,0,1,1,0,0,0,1.5873,1.5873
4,SPO,training_candidate,47,50,0,0,0,0,6,6,0,0,0,12.7660,12.7660
5,SZH,external_only,1445,2195,0,0,0,0,66,68,0,0,0,4.5675,4.7059


Combined site-level shape: (4280, 11)
Saved: outputs/tables/charged_site_level_quality_audit.csv


## ۱-۴. بررسی ساختار زمانی متغیر تقاضای شارژ

فایل `duration.csv` برای هر شهر بررسی می‌شود تا مشخص شود متغیر هدف چگونه در طول زمان ذخیره شده است، ستون زمانی چیست و آیا تعداد ستون‌های ایستگاه با فایل `sites.csv` سازگار است یا نه.

در این مرحله، فقط نمونه‌ای کوچک از هر فایل خوانده می‌شود و هیچ فایل زمانیِ بزرگ به‌طور کامل در حافظه بارگذاری نخواهد شد.

In [8]:
# Prompt: Inspect the schema and first records of hourly charging-duration files without loading complete time series.

duration_schema_rows = []
duration_samples = {}

with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
    for city in base_city_codes:
        archive_path = f"{city}/duration.csv"

        with charged_archive.open(archive_path) as raw_file:
            duration_sample = pd.read_csv(raw_file, nrows=5)

        duration_samples[city] = duration_sample.copy()

        time_column = duration_sample.columns[0]
        demand_columns = duration_sample.columns[1:]

        city_site_ids = (
            charged_sites
            .loc[charged_sites["city_code"] == city, "site_id"]
            .astype(str)
            .tolist()
        )

        demand_column_names = demand_columns.astype(str).tolist()

        duration_schema_rows.append(
            {
                "city_code": city,
                "dataset_role": (
                    "external_only"
                    if city == external_city_code
                    else "training_candidate"
                ),
                "sample_rows_read": len(duration_sample),
                "total_columns": len(duration_sample.columns),
                "time_column": time_column,
                "demand_column_count": len(demand_columns),
                "site_count": len(city_site_ids),
                "matching_site_column_count": len(
                    set(demand_column_names).intersection(set(city_site_ids))
                ),
                "first_demand_columns": " | ".join(demand_column_names[:5]),
            }
        )

duration_schema = pd.DataFrame(duration_schema_rows)

display(duration_schema)

print("Sample: AMS/duration.csv")
display(duration_samples["AMS"])

print("Sample: SZH/duration.csv")
display(duration_samples["SZH"])

duration_schema.to_csv(
    OUTPUT_TABLES_DIR / "charged_duration_schema_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: outputs/tables/charged_duration_schema_audit.csv")

,city_code,dataset_role,sample_rows_read,total_columns,time_column,demand_column_count,site_count,matching_site_column_count,first_demand_columns
0,AMS,training_candidate,5,2450,Unnamed: 0,2449,2449,2449,0 | 1 | 10 | 100 | 1000
1,JHB,training_candidate,5,48,Unnamed: 0,47,47,47,0 | 1 | 10 | 11 | 12
2,LOA,training_candidate,5,230,Unnamed: 0,229,229,229,0 | 1 | 10 | 100 | 101
3,MEL,training_candidate,5,64,Unnamed: 0,63,63,63,0 | 1 | 10 | 11 | 12
4,SPO,training_candidate,5,48,Unnamed: 0,47,47,47,0 | 1 | 10 | 11 | 12
5,SZH,external_only,5,1446,Unnamed: 0,1445,1445,1445,0 | 1 | 10 | 100 | 1000


Sample: AMS/duration.csv


Unnamed: 0      0      1     10    100   1000   1001   1002  \
0  2023-04-01 00:00:00 0.0000 0.0000 0.8333 0.0000 0.0000 0.0000 0.0000   
1  2023-04-01 01:00:00 0.0000 0.0000 1.0833 0.0000 0.0000 0.0000 0.0000   
2  2023-04-01 02:00:00 0.0000 0.0000 1.0000 0.0000 0.0000 0.0000 0.0000   
3  2023-04-01 03:00:00 0.0000 0.0000 1.0000 0.0000 0.0000 0.0000 0.0000   
4  2023-04-01 04:00:00 0.0000 0.0000 1.0000 0.0000 0.0000 0.0000 0.0000   

    1003   1004   1005   1006   1007   1008   1009    101   1010   1011  \
0 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.4360 0.0000 0.0000   
1 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.4408 0.0000 0.0000   
2 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.4457 0.0000 0.0000   
3 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.4505 0.0000 0.0000   
4 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.4553 0.0000 0.0000   

    1012   1013   1014   1015   1016   1017   1018   1019    102   1020  \
0 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
1 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
2 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
3 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
4 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   

    1021   1022   1023   1024   1025   1026   1027   1028   1029    103  \
0 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.1635   
1 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.1688   
2 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.1741   
3 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.1795   
4 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.1848   

    1030   1031   1032   1033   1034   1035   1036   1037   1038   1039  \
0 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
1 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
2 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
3 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
4 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   

     104   1040   1041   1042   1043   1044   1045   1046   1047   1048  \
0 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
1 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
2 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
3 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
4 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   

    1049    105   1050   1051   1052   1053   1054   1055   1056   1057  \
0 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
1 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
2 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
3 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
4 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   

    1058   1059    106   1060   1061   1062   1063   1064   1065   1066  \
0 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
1 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
2 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
3 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
4 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   

    1067   1068   1069    107   1070   1071   1072   1073   1074   1075  \
0 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
1 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
2 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
3 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000   
4 0.0000 0.0000 0.0000 0.0000

Sample: SZH/duration.csv


,Unnamed: 0,0,1,10,100,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,101,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,102,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,103,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039,104,1040,1041,1042,1043,1044,1045,1046,1047,1048,1049,105,1050,1051,1052,1053,1054,1055,1056,1057,1058,1059,106,1060,1061,1062,1063,1064,1065,1066,1067,1068,1069,107,1070,1071,1072,1073,1074,1075,1076,1077,1078,1079,108,1080,1081,1082,1083,1084,1085,1086,1087,1088,1089,109,1090,1091,1092,1093,1094,1095,1096,1097,1098,1099,11,110,1100,1101,1102,1103,1104,1105,1106,1107,1108,1109,111,1110,1111,1112,1113,1114,1115,1116,1117,1118,1119,112,1120,1121,1122,1123,1124,1125,1126,1127,1128,1129,113,1130,1131,1132,1133,1134,1135,1136,1137,1138,1139,114,1140,1141,1142,1143,1144,1145,1146,1147,1148,1149,115,1150,1151,1152,1153,1154,1155,1156,1157,1158,1159,116,1160,1161,1162,1163,1164,1165,1166,1167,1168,1169,117,1170,1171,1172,1173,1174,1175,1176,1177,1178,1179,118,1180,1181,1182,1183,1184,1185,1186,1187,1188,1189,119,1190,1191,1192,1193,1194,1195,1196,1197,1198,1199,12,120,1200,1201,1202,1203,1204,1205,1206,1207,1208,1209,121,1210,1211,1212,1213,1214,1215,1216,1217,1218,1219,122,1220,1221,1222,1223,1224,1225,1226,1227,1228,1229,123,1230,1231,1232,1233,1234,1235,1236,1237,1238,1239,124,1240,1241,1242,1243,1244,1245,1246,1247,1248,1249,125,1250,1251,1252,1253,1254,1255,1256,1257,1258,1259,126,1260,1261,1262,1263,1264,1265,1266,1267,1268,1269,127,1270,1271,1272,1273,1274,1275,1276,1277,1278,1279,128,1280,1281,1282,1283,1284,1285,1286,1287,1288,1289,129,1290,1291,1292,1293,1294,1295,1296,1297,1298,1299,13,130,1300,1301,1302,1303,1304,1305,1306,1307,1308,1309,131,1310,1311,1312,1313,1314,1315,1316,1317,1318,1319,132,1320,1321,1322,1323,1324,1325,1326,1327,1328,1329,133,1330,1331,1332,1333,1334,1335,1336,1337,1338,1339,134,1340,1341,1342,1343,1344,1345,1346,1347,1348,1349,135,1350,1351,1352,1353,1354,1355,1356,1357,1358,1359,136,1360,1361,1362,1363,1364,1365,1366,1367,1368,1369,137,1370,1371,1372,1373,1374,1375,1376,1377,1378,1379,138,1380,1381,1382,1383,1384,1385,1386,1387,1388,1389,139,1390,1391,1392,1393,1394,1395,1396,1397,1398,1399,14,140,1400,1401,1402,1403,1404,1405,1406,1407,1408,1409,141,1410,1411,1412,1413,1414,1415,1416,1417,1418,1419,142,1420,1421,1422,1423,1424,1425,1426,1427,1428,1429,143,1430,1431,1432,1433,1434,1435,1436,1437,1438,1439,144,1440,1441,1442,1443,1444,145,146,147,148,149,15,150,151,152,153,154,155,156,157,158,159,16,160,161,162,163,164,165,166,167,168,169,17,170,171,172,173,174,175,176,177,178,179,18,180,181,182,183,184,185,186,187,188,189,19,190,191,192,193,194,195,196,197,198,199,2,20,200,201,202,203,204,205,206,207,208,209,21,210,211,212,213,214,215,216,217,218,219,22,220,221,222,223,224,225,226,227,228,229,23,230,231,232,233,234,235,236,237,238,239,24,240,241,242,243,244,245,246,247,248,249,25,250,251,252,253,254,255,256,257,258,259,26,260,261,262,263,264,265,266,267,268,269,27,270,271,272,273,274,275,276,277,278,279,28,280,281,282,283,284,285,286,287,288,289,29,290,291,292,293,294,295,296,297,298,299,3,30,300,301,302,303,304,305,306,307,308,309,31,310,311,312,313,314,315,316,317,318,319,32,320,321,322,323,324,325,326,327,328,329,33,330,331,332,333,334,335,336,337,338,339,34,340,341,342,343,344,345,346,347,348,349,35,350,351,352,353,354,355,356,357,358,359,36,360,361,362,363,364,365,366,367,368,369,37,370,371,372,373,374,375,376,377,378,379,38,380,381,382,383,384,385,386,387,388,389,39,390,391,392,393,394,395,396,397,398,399,4,40,400,401,402,403,404,405,406,407,408,409,41,410,411,412,413,414,415,416,417,418,419,42,420,421,422,423,424,425,426,427,428,429,43,430,431,432,433,434,435,436,437,438,439,44,440,441,442,443,444,445,446,447,448,449,45,450,451,452,453,454,455,456,457,458,459,46,460,461,462,463,464,465,466,467,468,469,47,470,471,472,473,474,475,476,477,478,479,48,480,481,482,483,484,485,486,487,488,489,49,490,491,492,493,494,495,496,497,498,499,5,50,500,501,502,503,504,505,506,

Saved: outputs/tables/charged_duration_schema_audit.csv


## ۱-۵. ممیزی زمانی و کیفیت متغیر هدف

در این مرحله، تمام فایل‌های `duration.csv` به‌صورت قطعه‌ای بررسی می‌شوند تا بدون بارگذاری کامل داده‌های بزرگ در حافظه، تعداد رکوردهای زمانی، بازهٔ زمانی، تکرار یا شکست زمانی، مقادیر گمشده، صفر و منفی در متغیر هدف ارزیابی شود.

In [9]:
# Prompt: Audit temporal consistency and duration-data quality in chunks without loading complete files into memory.

CHUNK_SIZE = 200
duration_quality_rows = []

with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
    for city in base_city_codes:
        archive_path = f"{city}/duration.csv"

        total_time_rows = 0
        invalid_timestamp_count = 0
        missing_demand_count = 0
        zero_demand_count = 0
        negative_demand_count = 0
        total_demand_cells = 0
        timestamps = []

        with charged_archive.open(archive_path) as raw_file:
            for duration_chunk in pd.read_csv(raw_file, chunksize=CHUNK_SIZE):
                timestamp_column = duration_chunk.columns[0]
                demand_data = duration_chunk.iloc[:, 1:]

                parsed_timestamps = pd.to_datetime(
                    duration_chunk[timestamp_column],
                    errors="coerce",
                )

                total_time_rows += len(duration_chunk)
                invalid_timestamp_count += parsed_timestamps.isna().sum()

                valid_timestamps = parsed_timestamps.dropna()
                timestamps.extend(valid_timestamps.tolist())

                missing_demand_count += demand_data.isna().sum().sum()
                zero_demand_count += (demand_data == 0).sum().sum()
                negative_demand_count += (demand_data < 0).sum().sum()
                total_demand_cells += demand_data.shape[0] * demand_data.shape[1]

        timestamp_index = pd.DatetimeIndex(timestamps).sort_values()

        time_differences = timestamp_index.to_series().diff().dropna()

        duration_quality_rows.append(
            {
                "city_code": city,
                "dataset_role": (
                    "external_only"
                    if city == external_city_code
                    else "training_candidate"
                ),
                "time_records": total_time_rows,
                "start_timestamp": timestamp_index.min(),
                "end_timestamp": timestamp_index.max(),
                "invalid_timestamps": invalid_timestamp_count,
                "duplicate_timestamps": timestamp_index.duplicated().sum(),
                "most_common_interval": (
                    time_differences.mode().iloc[0]
                    if not time_differences.empty
                    else pd.NaT
                ),
                "irregular_time_gaps": (
                    (time_differences != time_differences.mode().iloc[0]).sum()
                    if not time_differences.empty
                    else 0
                ),
                "missing_demand_values": missing_demand_count,
                "zero_demand_values": zero_demand_count,
                "zero_demand_percent": (
                    100 * zero_demand_count / total_demand_cells
                ),
                "negative_demand_values": negative_demand_count,
            }
        )

duration_quality_audit = pd.DataFrame(duration_quality_rows)

display(duration_quality_audit)

duration_quality_audit.to_csv(
    OUTPUT_TABLES_DIR / "charged_duration_temporal_quality_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: outputs/tables/charged_duration_temporal_quality_audit.csv")

,city_code,dataset_role,time_records,start_timestamp,end_timestamp,invalid_timestamps,duplicate_timestamps,most_common_interval,irregular_time_gaps,missing_demand_values,zero_demand_values,zero_demand_percent,negative_demand_values
0,AMS,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0,0 days 01:00:00,0,0,6060792,56.3480,0
1,JHB,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0,0 days 01:00:00,0,0,72984,35.3564,0
2,LOA,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0,0 days 01:00:00,0,0,219670,21.8410,0
3,MEL,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0,0 days 01:00:00,0,0,82397,29.7789,0
4,SPO,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0,0 days 01:00:00,0,0,58384,28.2835,0
5,SZH,external_only,4392,2023-04-01,2023-09-30 23:00:00,0,0,0 days 01:00:00,0,0,431810,6.8040,0


Saved: outputs/tables/charged_duration_temporal_quality_audit.csv


## ۱-۶. بررسی منابع ویژگی‌های قابل‌انتقال

فایل‌های نقاط مورد علاقه (`poi.csv`)، شرایط جوی (`weather.csv`) و مشخصات شارژرها (`chargers.csv`) بررسی می‌شوند تا متغیرهایی شناسایی شوند که بتوان آن‌ها را به‌صورت مفهومی برای شهر اصفهان نیز استخراج یا برآورد کرد.

In [10]:
# Prompt: Inspect the schema of POI, weather, and charger-information files across all CHARGED cities.

PORTABLE_FEATURE_FILES = [
    "poi.csv",
    "weather.csv",
    "chargers.csv",
]

feature_schema_rows = []
feature_samples = {}

with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
    for city in base_city_codes:
        for file_name in PORTABLE_FEATURE_FILES:
            archive_path = f"{city}/{file_name}"

            with charged_archive.open(archive_path) as raw_file:
                feature_sample = pd.read_csv(raw_file, nrows=5)

            feature_samples[(city, file_name)] = feature_sample.copy()

            feature_schema_rows.append(
                {
                    "city_code": city,
                    "dataset_role": (
                        "external_only"
                        if city == external_city_code
                        else "training_candidate"
                    ),
                    "file_name": file_name,
                    "sample_rows_read": len(feature_sample),
                    "column_count": len(feature_sample.columns),
                    "columns": " | ".join(feature_sample.columns.astype(str)),
                }
            )

feature_schema = pd.DataFrame(feature_schema_rows)

display(feature_schema)

print("Sample: AMS/poi.csv")
display(feature_samples[("AMS", "poi.csv")])

print("Sample: AMS/weather.csv")
display(feature_samples[("AMS", "weather.csv")])

print("Sample: AMS/chargers.csv")
display(feature_samples[("AMS", "chargers.csv")])

feature_schema.to_csv(
    OUTPUT_TABLES_DIR / "charged_portable_feature_schema.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: outputs/tables/charged_portable_feature_schema.csv")

,city_code,dataset_role,file_name,sample_rows_read,column_count,columns
0,AMS,training_candidate,poi.csv,5,4,Unnamed: 0 | type | longitude | latitude
1,AMS,training_candidate,weather.csv,5,19,time | temp | feelslike | humidity | dew | pre...
2,AMS,training_candidate,chargers.csv,5,7,charger_id | longitude | latitude | site_id | ...
3,JHB,training_candidate,poi.csv,5,4,Unnamed: 0 | type | longitude | latitude
4,JHB,training_candidate,weather.csv,5,19,time | temp | feelslike | humidity | dew | pre...
5,JHB,training_candidate,chargers.csv,5,7,charger_id | longitude | latitude | site_id | ...
6,LOA,training_candidate,poi.csv,5,4,Unnamed: 0 | type | longitude | latitude
7,LOA,training_candidate,weather.csv,5,19,time | temp | feelslike | humidity | dew | pre...
8,LOA,training_candidate,chargers.csv,5,7,charger_id | longitude | latitude | site | tot...
9,MEL,training_candidate,poi.csv,5,4,Unnamed: 0 | type | longitude | latitude


Sample: AMS/poi.csv


,Unnamed: 0,type,longitude,latitude
0,0,university,4.9115,52.3638
1,1,other,4.9227,52.3628
2,2,fire_station,4.9292,52.3605
3,3,place_of_worship,4.9131,52.3833
4,4,school,4.9132,52.3813


Sample: AMS/weather.csv


,time,temp,feelslike,humidity,dew,precip,snow,snowdepth,preciptype,windgust,windspeed,winddir,pressure,visibility,cloudcover,solarradiation,solarenergy,uvindex,conditions
0,2023/4/1 0:00,9.6000,7.2000,98.7900,9.4000,0.0000,0,0,NaN,25.2000,17.2000,211,988.9000,4.5000,100,0,0,0,1
1,2023/4/1 1:00,9.3000,6.6000,98.6300,9.1000,0.0000,0,0,NaN,27.0000,19.6000,217,989.3000,5.4000,100,0,0,0,1
2,2023/4/1 2:00,9.0000,6.5000,99.3200,8.9000,1.0140,0,0,['rain'],26.4000,16.0000,223,989.6000,3.7000,100,0,0,0,5
3,2023/4/1 3:00,8.9000,6.5000,98.5000,8.7000,0.2710,0,0,['rain'],19.8000,16.4000,241,989.9000,2.2000,100,0,0,0,5
4,2023/4/1 4:00,9.2000,6.6000,99.0200,9.0000,0.1350,0,0,['rain'],24.8000,17.6000,302,990.4000,3.9000,100,0,0,0,5


Sample: AMS/chargers.csv


,charger_id,longitude,latitude,site_id,total_duration,total_volume,avg_power
0,1000592625,4.8353,52.4059,0,0.0000,0.0000,0.0000
1,1000598195,4.8586,52.4099,1,0.0000,0.0000,0.0000
2,1000598735,4.8771,52.3399,2,0.0000,0.0000,0.0000
3,1000617235,4.9086,52.3696,3,0.0000,0.0000,0.0000
4,1000619895,4.9086,52.3696,3,0.0000,0.0000,0.0000


Saved: outputs/tables/charged_portable_feature_schema.csv


## ۱-۷. ممیزی کیفیت نقاط مورد علاقه، آب‌وهوا و پورت‌های شارژ

در این مرحله، کیفیت سه منبع ویژگی قابل‌انتقال بررسی می‌شود: اعتبار مختصات و تنوع نوع POIها، پیوستگی و مقادیر گمشدهٔ داده‌های جوی، و سازگاری اتصال پورت‌های شارژ به ایستگاه‌ها.

متغیرهای تجمعی تقاضا در فایل `chargers.csv` فقط برای کنترل کیفیت استفاده می‌شوند و به‌عنوان ویژگی پیش‌بینی‌کننده وارد مدل نخواهند شد.

In [11]:
# Prompt: Audit POI, weather, and charger-linkage quality across CHARGED cities.

portable_quality_rows = []
poi_type_rows = []

for city in base_city_codes:
    city_poi = read_csv_from_charged_archive(city, "poi.csv").copy()
    city_weather = read_csv_from_charged_archive(city, "weather.csv").copy()
    city_chargers = read_csv_from_charged_archive(city, "chargers.csv").copy()

    city_sites = charged_sites.loc[
        charged_sites["city_code"] == city
    ].copy()

    charger_site_column = (
        "site_id" if "site_id" in city_chargers.columns else "site"
    )

    city_weather["timestamp"] = pd.to_datetime(
        city_weather["time"],
        errors="coerce",
    )

    valid_site_ids = set(city_sites["site_id"].astype(str))
    charger_site_ids = city_chargers[charger_site_column].astype(str)

    portable_quality_rows.append(
        {
            "city_code": city,
            "dataset_role": (
                "external_only"
                if city == external_city_code
                else "training_candidate"
            ),
            "poi_count": len(city_poi),
            "poi_type_count": city_poi["type"].nunique(dropna=True),
            "poi_missing_type": city_poi["type"].isna().sum(),
            "poi_missing_coordinates": (
                city_poi["longitude"].isna().sum()
                + city_poi["latitude"].isna().sum()
            ),
            "poi_invalid_coordinates": (
                ((city_poi["longitude"] < -180) | (city_poi["longitude"] > 180)).sum()
                + ((city_poi["latitude"] < -90) | (city_poi["latitude"] > 90)).sum()
            ),
            "weather_records": len(city_weather),
            "weather_invalid_timestamps": city_weather["timestamp"].isna().sum(),
            "weather_duplicate_timestamps": city_weather["timestamp"].duplicated().sum(),
            "weather_missing_values": city_weather.isna().sum().sum(),
            "charger_count": len(city_chargers),
            "chargers_missing_coordinates": (
                city_chargers["longitude"].isna().sum()
                + city_chargers["latitude"].isna().sum()
            ),
            "chargers_without_valid_site": (
                ~charger_site_ids.isin(valid_site_ids)
            ).sum(),
        }
    )

    city_poi_type_counts = (
        city_poi["type"]
        .value_counts(dropna=False)
        .rename_axis("poi_type")
        .reset_index(name="poi_count")
    )

    city_poi_type_counts.insert(0, "city_code", city)
    poi_type_rows.append(city_poi_type_counts)

portable_quality_audit = pd.DataFrame(portable_quality_rows)
poi_type_distribution = pd.concat(poi_type_rows, ignore_index=True)

display(portable_quality_audit)

print("Most frequent POI types across all training cities:")
display(
    poi_type_distribution
    .loc[poi_type_distribution["city_code"].isin(training_city_codes)]
    .groupby("poi_type", as_index=False)["poi_count"]
    .sum()
    .sort_values("poi_count", ascending=False)
    .head(20)
)

portable_quality_audit.to_csv(
    OUTPUT_TABLES_DIR / "charged_portable_feature_quality_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

poi_type_distribution.to_csv(
    OUTPUT_TABLES_DIR / "charged_poi_type_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved portable-feature audit tables in outputs/tables/")

,city_code,dataset_role,poi_count,poi_type_count,poi_missing_type,poi_missing_coordinates,poi_invalid_coordinates,weather_records,weather_invalid_timestamps,weather_duplicate_timestamps,weather_missing_values,charger_count,chargers_missing_coordinates,chargers_without_valid_site
0,AMS,training_candidate,51340,157,0,0,0,4392,0,0,3968,3526,0,0
1,JHB,training_candidate,47488,112,0,0,0,4392,0,0,4354,61,0,0
2,LOA,training_candidate,42782,148,0,0,0,4392,0,0,4321,506,0,0
3,MEL,training_candidate,617806,213,0,0,0,4392,0,0,4147,64,0,0
4,SPO,training_candidate,49719,133,0,0,0,4392,0,0,4341,50,0,0
5,SZH,external_only,47181,91,0,0,0,4392,0,0,3977,2195,0,0


Most frequent POI types across all training cities:


,poi_type,poi_count
204,other,626380
215,parking_space,51274
208,parking,37772
39,bench,12141
251,restaurant,10183
128,fast_food,5626
40,bicycle_parking,5064
263,school,4925
62,cafe,4584
324,waste_basket,3978


Saved portable-feature audit tables in outputs/tables/


## ۱-۸. بررسی دقیق مقادیر گمشده و پیوستگی داده‌های جوی

برای جلوگیری از تصمیم‌گیری بر اساس مجموع کلی مقادیر گمشده، الگوی گمشدگی هر متغیر جوی به تفکیک شهر بررسی می‌شود. همچنین پیوستگی زمانی فایل‌های جوی کنترل می‌شود تا هم‌ترازی آن‌ها با دادهٔ ساعتی تقاضای شارژ تأیید شود.

In [12]:
# Prompt: Identify weather-data missingness by variable and verify hourly temporal continuity.

weather_audit_rows = []
weather_missingness_rows = []

for city in base_city_codes:
    city_weather = read_csv_from_charged_archive(city, "weather.csv").copy()

    city_weather["timestamp"] = pd.to_datetime(
        city_weather["time"],
        errors="coerce",
    )

    weather_timestamps = city_weather["timestamp"].dropna().sort_values()
    weather_intervals = weather_timestamps.diff().dropna()

    weather_audit_rows.append(
        {
            "city_code": city,
            "dataset_role": (
                "external_only"
                if city == external_city_code
                else "training_candidate"
            ),
            "weather_records": len(city_weather),
            "start_timestamp": weather_timestamps.min(),
            "end_timestamp": weather_timestamps.max(),
            "duplicate_timestamps": weather_timestamps.duplicated().sum(),
            "most_common_interval": (
                weather_intervals.mode().iloc[0]
                if not weather_intervals.empty
                else pd.NaT
            ),
            "irregular_time_gaps": (
                (weather_intervals != weather_intervals.mode().iloc[0]).sum()
                if not weather_intervals.empty
                else 0
            ),
        }
    )

    for column_name in city_weather.columns:
        if column_name != "timestamp":
            missing_count = city_weather[column_name].isna().sum()

            weather_missingness_rows.append(
                {
                    "city_code": city,
                    "dataset_role": (
                        "external_only"
                        if city == external_city_code
                        else "training_candidate"
                    ),
                    "variable": column_name,
                    "missing_count": missing_count,
                    "missing_percent": 100 * missing_count / len(city_weather),
                }
            )

weather_temporal_audit = pd.DataFrame(weather_audit_rows)
weather_missingness = pd.DataFrame(weather_missingness_rows)

display(weather_temporal_audit)

print("Variables with missing values in training cities:")
display(
    weather_missingness
    .loc[
        (weather_missingness["city_code"].isin(training_city_codes))
        & (weather_missingness["missing_count"] > 0)
    ]
    .sort_values(
        ["missing_count", "city_code"],
        ascending=[False, True],
    )
)

weather_temporal_audit.to_csv(
    OUTPUT_TABLES_DIR / "charged_weather_temporal_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

weather_missingness.to_csv(
    OUTPUT_TABLES_DIR / "charged_weather_missingness_by_variable.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved weather audit tables in outputs/tables/")

,city_code,dataset_role,weather_records,start_timestamp,end_timestamp,duplicate_timestamps,most_common_interval,irregular_time_gaps
0,AMS,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0 days 01:00:00,0
1,JHB,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0 days 01:00:00,0
2,LOA,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0 days 01:00:00,0
3,MEL,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0 days 01:00:00,0
4,SPO,training_candidate,4392,2023-04-01,2023-09-30 23:00:00,0,0 days 01:00:00,0
5,SZH,external_only,4392,2023-04-01,2023-09-30 23:00:00,0,0 days 01:00:00,0


Variables with missing values in training cities:


,city_code,dataset_role,variable,missing_count,missing_percent
27,JHB,training_candidate,preciptype,4354,99.1348
84,SPO,training_candidate,preciptype,4341,98.8388
46,LOA,training_candidate,preciptype,4321,98.3834
65,MEL,training_candidate,preciptype,4147,94.4217
8,AMS,training_candidate,preciptype,3968,90.3461


Saved weather audit tables in outputs/tables/


## ۱-۹. تفسیر گمشدگی نوع بارش

متغیر `preciptype` دارای گمشدگی گسترده است. در این مرحله بررسی می‌شود که آیا این گمشدگی عمدتاً با مقدار صفر در متغیر کمی `precip` هم‌زمان است یا خیر. نتیجهٔ این بررسی تعیین می‌کند که آیا متغیر نوع بارش کنار گذاشته شود و متغیر قابل‌اعتمادتر «وجود بارش» جایگزین آن شود.

In [13]:
# Prompt: Determine whether missing precipitation-type values correspond to no recorded precipitation.

precipitation_type_audit_rows = []

for city in base_city_codes:
    city_weather = read_csv_from_charged_archive(city, "weather.csv").copy()

    preciptype_missing = city_weather["preciptype"].isna()
    precipitation_amount = pd.to_numeric(
        city_weather["precip"],
        errors="coerce",
    )

    precipitation_type_audit_rows.append(
        {
            "city_code": city,
            "dataset_role": (
                "external_only"
                if city == external_city_code
                else "training_candidate"
            ),
            "missing_preciptype": preciptype_missing.sum(),
            "missing_preciptype_with_zero_precip": (
                preciptype_missing & (precipitation_amount == 0)
            ).sum(),
            "missing_preciptype_with_positive_precip": (
                preciptype_missing & (precipitation_amount > 0)
            ).sum(),
            "missing_preciptype_with_missing_precip": (
                preciptype_missing & precipitation_amount.isna()
            ).sum(),
            "non_missing_preciptype_values": (
                city_weather.loc[
                    city_weather["preciptype"].notna(),
                    "preciptype",
                ]
                .astype(str)
                .unique()
                .tolist()
            ),
        }
    )

precipitation_type_audit = pd.DataFrame(
    precipitation_type_audit_rows
)

display(precipitation_type_audit)

precipitation_type_audit.to_csv(
    OUTPUT_TABLES_DIR / "charged_preciptype_missingness_interpretation.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved: outputs/tables/charged_preciptype_missingness_interpretation.csv"
)

,city_code,dataset_role,missing_preciptype,missing_preciptype_with_zero_precip,missing_preciptype_with_positive_precip,missing_preciptype_with_missing_precip,non_missing_preciptype_values
0,AMS,training_candidate,3968,3968,0,0,[['rain']]
1,JHB,training_candidate,4354,4354,0,0,[['rain']]
2,LOA,training_candidate,4321,4321,0,0,[['rain']]
3,MEL,training_candidate,4147,4147,0,0,[['rain']]
4,SPO,training_candidate,4341,4341,0,0,[['rain']]
5,SZH,external_only,3977,3977,0,0,[['rain']]


Saved: outputs/tables/charged_preciptype_missingness_interpretation.csv


## ۱-۱۰. کنترل سازگاری بین فایل‌های زمانی و مکانی

در این مرحله، مجموع مدت شارژ ساعتی هر ایستگاه در `duration.csv` با مقدار تجمعی همان ایستگاه در `sites.csv` مقایسه می‌شود. این کنترل، صحت اتصال فایل‌های زمانی و مکانی و قابلیت استفاده از متغیر هدف را تأیید می‌کند.

In [14]:
# Prompt: Verify that site-level total duration matches the sum of hourly duration records.

duration_consistency_rows = []

with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
    for city in base_city_codes:
        archive_path = f"{city}/duration.csv"

        hourly_duration_sum = None

        with charged_archive.open(archive_path) as raw_file:
            for duration_chunk in pd.read_csv(raw_file, chunksize=CHUNK_SIZE):
                numeric_duration = duration_chunk.iloc[:, 1:].apply(
                    pd.to_numeric,
                    errors="coerce",
                )

                chunk_sum = numeric_duration.sum(axis=0, min_count=1)

                if hourly_duration_sum is None:
                    hourly_duration_sum = chunk_sum
                else:
                    hourly_duration_sum = hourly_duration_sum.add(
                        chunk_sum,
                        fill_value=0,
                    )

        city_sites = charged_sites.loc[
            charged_sites["city_code"] == city,
            ["site_id", "total_duration"],
        ].copy()

        city_sites["site_id"] = city_sites["site_id"].astype(str)
        city_sites["total_duration"] = pd.to_numeric(
            city_sites["total_duration"],
            errors="coerce",
        )

        hourly_duration_sum.index = hourly_duration_sum.index.astype(str)

        consistency_comparison = (
            city_sites
            .set_index("site_id")
            .rename(columns={"total_duration": "sites_total_duration"})
            .join(
                hourly_duration_sum.rename("hourly_total_duration"),
                how="outer",
            )
        )

        consistency_comparison["absolute_difference"] = (
            consistency_comparison["sites_total_duration"]
            - consistency_comparison["hourly_total_duration"]
        ).abs()

        comparable_records = consistency_comparison.dropna(
            subset=["sites_total_duration", "hourly_total_duration"]
        )

        matched_records = np.isclose(
            comparable_records["sites_total_duration"],
            comparable_records["hourly_total_duration"],
            rtol=1e-5,
            atol=1e-4,
        )

        duration_consistency_rows.append(
            {
                "city_code": city,
                "dataset_role": (
                    "external_only"
                    if city == external_city_code
                    else "training_candidate"
                ),
                "site_records": len(city_sites),
                "hourly_duration_columns": len(hourly_duration_sum),
                "comparable_sites": len(comparable_records),
                "matching_sites": matched_records.sum(),
                "non_matching_sites": (~matched_records).sum(),
                "max_absolute_difference": (
                    comparable_records["absolute_difference"].max()
                ),
                "mean_absolute_difference": (
                    comparable_records["absolute_difference"].mean()
                ),
            }
        )

duration_consistency_audit = pd.DataFrame(duration_consistency_rows)

display(duration_consistency_audit)

duration_consistency_audit.to_csv(
    OUTPUT_TABLES_DIR / "charged_duration_cross_file_consistency_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved: outputs/tables/charged_duration_cross_file_consistency_audit.csv"
)

,city_code,dataset_role,site_records,hourly_duration_columns,comparable_sites,matching_sites,non_matching_sites,max_absolute_difference,mean_absolute_difference
0,AMS,training_candidate,2449,2449,2449,1061,1388,"298,216.0000","4,254.6989"
1,JHB,training_candidate,47,47,47,12,35,"4,734.0634","1,029.0840"
2,LOA,training_candidate,229,229,229,5,224,"52,244.4934","3,904.7093"
3,MEL,training_candidate,63,63,63,1,62,"18,584.3467","5,202.2427"
4,SPO,training_candidate,47,47,47,6,41,"2,184.4398",795.6947
5,SZH,external_only,1445,1445,1445,66,1379,"1,392,114.5833","21,484.7223"


Saved: outputs/tables/charged_duration_cross_file_consistency_audit.csv


## ۱-۱۱. تحلیل اختلاف شاخص‌های تجمعی تقاضا

نتایج کنترل سازگاری نشان داد که مقادیر تجمعی `total_duration` در فایل `sites.csv` برای ایستگاه‌های فعال با مجموع دادهٔ ساعتی `duration.csv` هم‌خوان نیست. در این مرحله، اندازه و الگوی اختلاف بررسی می‌شود.

از این پس، دادهٔ ساعتی `duration.csv` تنها منبع معتبر برای ساخت متغیر هدف خواهد بود.

In [15]:
# Prompt: Diagnose the relationship between site-level total duration and sums derived from hourly duration data.

duration_difference_rows = []
ams_difference_sample = None

with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
    for city in base_city_codes:
        archive_path = f"{city}/duration.csv"

        hourly_duration_sum = None

        with charged_archive.open(archive_path) as raw_file:
            for duration_chunk in pd.read_csv(raw_file, chunksize=CHUNK_SIZE):
                numeric_duration = duration_chunk.iloc[:, 1:].apply(
                    pd.to_numeric,
                    errors="coerce",
                )

                chunk_sum = numeric_duration.sum(axis=0, min_count=1)

                if hourly_duration_sum is None:
                    hourly_duration_sum = chunk_sum
                else:
                    hourly_duration_sum = hourly_duration_sum.add(
                        chunk_sum,
                        fill_value=0,
                    )

        city_sites = charged_sites.loc[
            charged_sites["city_code"] == city,
            ["site_id", "total_duration"],
        ].copy()

        city_sites["site_id"] = city_sites["site_id"].astype(str)
        city_sites["sites_total_duration"] = pd.to_numeric(
            city_sites["total_duration"],
            errors="coerce",
        )

        hourly_duration_sum.index = hourly_duration_sum.index.astype(str)

        comparison = (
            city_sites
            .set_index("site_id")[["sites_total_duration"]]
            .join(
                hourly_duration_sum.rename("hourly_total_duration"),
                how="inner",
            )
        )

        active_comparison = comparison.loc[
            (comparison["sites_total_duration"] > 0)
            & (comparison["hourly_total_duration"] > 0)
        ].copy()

        active_comparison["hourly_to_sites_ratio"] = (
            active_comparison["hourly_total_duration"]
            / active_comparison["sites_total_duration"]
        )

        duration_difference_rows.append(
            {
                "city_code": city,
                "active_comparable_sites": len(active_comparison),
                "duration_correlation": active_comparison[
                    "sites_total_duration"
                ].corr(active_comparison["hourly_total_duration"]),
                "ratio_min": active_comparison[
                    "hourly_to_sites_ratio"
                ].min(),
                "ratio_median": active_comparison[
                    "hourly_to_sites_ratio"
                ].median(),
                "ratio_mean": active_comparison[
                    "hourly_to_sites_ratio"
                ].mean(),
                "ratio_max": active_comparison[
                    "hourly_to_sites_ratio"
                ].max(),
            }
        )

        if city == "AMS":
            ams_difference_sample = (
                active_comparison
                .assign(
                    absolute_difference=lambda dataframe: (
                        dataframe["sites_total_duration"]
                        - dataframe["hourly_total_duration"]
                    ).abs()
                )
                .sort_values("absolute_difference", ascending=False)
                .head(10)
                .reset_index()
            )

duration_difference_audit = pd.DataFrame(duration_difference_rows)

display(duration_difference_audit)

print("Ten largest duration differences in Amsterdam:")
display(ams_difference_sample)

duration_difference_audit.to_csv(
    OUTPUT_TABLES_DIR / "charged_duration_difference_diagnosis.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: outputs/tables/charged_duration_difference_diagnosis.csv")

,city_code,active_comparable_sites,duration_correlation,ratio_min,ratio_median,ratio_mean,ratio_max
0,AMS,1388,0.9851,0.3733,0.6692,0.6634,0.9168
1,JHB,35,0.9781,0.3401,0.4756,0.4784,0.7524
2,LOA,224,0.9926,0.2222,0.5281,0.5286,0.7774
3,MEL,62,0.9721,0.3849,0.5285,0.5290,0.6778
4,SPO,41,0.9916,0.4324,0.5393,0.5371,0.6468
5,SZH,1379,0.9926,0.2221,0.5014,0.5092,0.8385


Ten largest duration differences in Amsterdam:


,index,sites_total_duration,hourly_total_duration,hourly_to_sites_ratio,absolute_difference
0,31,"697,759.8174","399,543.8174",0.5726,"298,216.0000"
1,25,"527,697.6250","254,645.9338",0.4826,"273,051.6912"
2,14,"345,961.1250","193,766.3342",0.5601,"152,194.7908"
3,18,"278,870.0417","135,166.3805",0.4847,"143,703.6612"
4,2268,"252,854.8371","128,201.0689",0.5070,"124,653.7682"
5,78,"245,873.5625","139,617.7292",0.5678,"106,255.8333"
6,2211,"227,480.0000","130,328.0000",0.5729,"97,152.0000"
7,2205,"227,480.0000","130,328.0000",0.5729,"97,152.0000"
8,679,"172,385.5567","100,748.7234",0.5844,"71,636.8333"
9,30,"152,463.2810","83,008.3491",0.5444,"69,454.9319"


Saved: outputs/tables/charged_duration_difference_diagnosis.csv


## ۱-۱۲. تصمیم‌های نهایی ممیزی دیتاست CHARGED

ممیزی نشان داد که داده‌های زمانی تقاضا، داده‌های مکانی ایستگاه‌ها، POIها و آب‌وهوا از نظر شناسه، مختصات و زمان باکیفیت و قابل‌استفاده‌اند.

با این حال، مقادیر تجمعی `total_duration` در فایل `sites.csv` با مجموع مقادیر ساعتی `duration.csv` سازگاری عددی ندارند. ازاین‌رو، در تمام مراحل بعدی، متغیر هدف فقط از داده‌های ساعتی `duration.csv` استخراج خواهد شد.

مقادیر صفر تقاضا حفظ می‌شوند، زیرا نشان‌دهندهٔ ساعات یا ایستگاه‌های بدون استفاده هستند. متغیر `preciptype` حذف خواهد شد و ویژگی جایگزین `has_precipitation` از مقدار `precip` ساخته می‌شود. دسته‌های خام POI نیز پیش از مدل‌سازی به گروه‌های مفهومی و قابل‌انتقال تبدیل خواهند شد.

In [17]:
# Prompt: Create and save a final city-level profile for the CHARGED audit.

city_info_rows = []

for city in base_city_codes:
    city_info = read_csv_from_charged_archive(city, "info.csv").copy()
    city_info["city_code"] = city
    city_info["dataset_role"] = (
        "external_only"
        if city == external_city_code
        else "training_candidate"
    )
    city_info_rows.append(city_info)

charged_city_info = pd.concat(city_info_rows, ignore_index=True)
charged_city_info = charged_city_info.rename(
    columns={"total_chargers": "reported_total_chargers"}
)
charged_city_profile = (
    charged_city_info
    .merge(
        site_level_audit[
            [
                "city_code",
                "site_count",
                "total_chargers",
                "zero_duration_percent",
            ]
        ],
        on="city_code",
        how="left",
    )
    .merge(
        duration_quality_audit[
            [
                "city_code",
                "time_records",
                "start_timestamp",
                "end_timestamp",
                "zero_demand_percent",
            ]
        ],
        on="city_code",
        how="left",
    )
    .merge(
        portable_quality_audit[
            [
                "city_code",
                "poi_count",
                "poi_type_count",
            ]
        ],
        on="city_code",
        how="left",
    )
    .merge(
        duration_difference_audit[
            [
                "city_code",
                "duration_correlation",
                "ratio_median",
            ]
        ],
        on="city_code",
        how="left",
    )
)

training_site_total = charged_city_profile.loc[
    charged_city_profile["dataset_role"] == "training_candidate",
    "site_count",
].sum()

charged_city_profile["training_site_share_percent"] = np.where(
    charged_city_profile["dataset_role"] == "training_candidate",
    100 * charged_city_profile["site_count"] / training_site_total,
    np.nan,
)

profile_columns = [
    "city_code",
    "city",
    "country",
    "dataset_role",
    "site_count",
    "total_chargers",
    "training_site_share_percent",
    "poi_count",
    "poi_type_count",
    "time_records",
    "zero_duration_percent",
    "zero_demand_percent",
    "duration_correlation",
    "ratio_median",
]

display(charged_city_profile[profile_columns])

charged_city_profile.to_csv(
    OUTPUT_TABLES_DIR / "charged_final_city_profile.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: outputs/tables/charged_final_city_profile.csv")

,city_code,city,country,dataset_role,site_count,total_chargers,training_site_share_percent,poi_count,poi_type_count,time_records,zero_duration_percent,zero_demand_percent,duration_correlation,ratio_median
0,AMS,Amsterdam,Netherlands,training_candidate,2449,3526,86.3845,51340,157,4392,43.3238,56.3480,0.9851,0.6692
1,JHB,Johannesburg,SouthAfrica,training_candidate,47,61,1.6578,47488,112,4392,25.5319,35.3564,0.9781,0.4756
2,LOA,UnitedStates,LOA,training_candidate,229,506,8.0776,42782,148,4392,2.1834,21.8410,0.9926,0.5281
3,MEL,Australia,MEL,training_candidate,63,64,2.2222,617806,213,4392,1.5873,29.7789,0.9721,0.5285
4,SPO,Brazil,SPO,training_candidate,47,50,1.6578,49719,133,4392,12.7660,28.2835,0.9916,0.5393
5,SZH,China,SZH,external_only,1445,2195,NaN,47181,91,4392,4.5675,6.8040,0.9926,0.5014


Saved: outputs/tables/charged_final_city_profile.csv


In [18]:
# Prompt: Standardize city and country labels using the verified CHARGED city codes.

city_lookup = pd.DataFrame(
    {
        "city_code": ["AMS", "JHB", "LOA", "MEL", "SPO", "SZH"],
        "city_name": [
            "Amsterdam",
            "Johannesburg",
            "Los Angeles",
            "Melbourne",
            "Sao Paulo",
            "Shenzhen",
        ],
        "country_name": [
            "Netherlands",
            "South Africa",
            "United States",
            "Australia",
            "Brazil",
            "China",
        ],
    }
)

charged_city_profile = (
    charged_city_profile
    .drop(columns=["city", "country"], errors="ignore")
    .merge(city_lookup, on="city_code", how="left")
)

profile_columns = [
    "city_code",
    "city_name",
    "country_name",
    "dataset_role",
    "site_count",
    "total_chargers",
    "training_site_share_percent",
    "poi_count",
    "poi_type_count",
    "time_records",
    "zero_duration_percent",
    "zero_demand_percent",
    "duration_correlation",
    "ratio_median",
]

display(charged_city_profile[profile_columns])

charged_city_profile.to_csv(
    OUTPUT_TABLES_DIR / "charged_final_city_profile.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Updated: outputs/tables/charged_final_city_profile.csv")

,city_code,city_name,country_name,dataset_role,site_count,total_chargers,training_site_share_percent,poi_count,poi_type_count,time_records,zero_duration_percent,zero_demand_percent,duration_correlation,ratio_median
0,AMS,Amsterdam,Netherlands,training_candidate,2449,3526,86.3845,51340,157,4392,43.3238,56.3480,0.9851,0.6692
1,JHB,Johannesburg,South Africa,training_candidate,47,61,1.6578,47488,112,4392,25.5319,35.3564,0.9781,0.4756
2,LOA,Los Angeles,United States,training_candidate,229,506,8.0776,42782,148,4392,2.1834,21.8410,0.9926,0.5281
3,MEL,Melbourne,Australia,training_candidate,63,64,2.2222,617806,213,4392,1.5873,29.7789,0.9721,0.5285
4,SPO,Sao Paulo,Brazil,training_candidate,47,50,1.6578,49719,133,4392,12.7660,28.2835,0.9916,0.5393
5,SZH,Shenzhen,China,external_only,1445,2195,NaN,47181,91,4392,4.5675,6.8040,0.9926,0.5014


Updated: outputs/tables/charged_final_city_profile.csv
